# BC2026 XC API Downloader **Part 3** (72 non-Aves species)

Part 1 (76 Aves) + Part 2 (83 Aves) で Aves 159 species 完了。
**Part 3 = 残り non-Aves 72 species (Amphibia 35 + Insecta 28 + Mammalia 8 + Reptilia 1)**

XC は鳥が main だが non-Aves 録音もあり、Amphibia/Mammalia 中心に取得期待。

## 変更点 (vs Part 2)
- species list: 72 non-Aves (Amphibia, Insecta, Mammalia, Reptilia)
- Output dir: `/kaggle/working/xc_api_part3/`
- Output Dataset: `birdclef2026-xc-api-aves-audio-part3`

## 期待

- 取得可能: Amphibia ~10-20、Mammalia 2-5、一部 Insecta、Reptilia 0-1
- 取得不能: XC に存在しない species 多数 (skip, no error)
- ~5-15 GB 想定

---


# BC2026 XC API Direct Downloader

XC API を直接 query して **BC2026 234 species の追加録音**を polite に DL する。

## 流れ

1. BC2026 `taxonomy.csv` から 234 species 取得
2. 各 species を XC API で query (pagination 対応)
3. Filter (quality A/B/C、length ≤120s、license non-ND)
4. Polite DL (1.5s delay/req、retry 3 回、resume 対応)
5. Kaggle Dataset 化

## politeness

| Setting | Value | Reason |
|---|---|---|
| Delay (API query) | 1.5 sec | XC API rate limit 配慮 |
| Delay (audio DL) | 1.5 sec | XC server load |
| Retry | 3 attempts | exponential backoff [5,15,60]s |
| User-Agent | identifying | XC ToS / 礼儀 |
| Resume | enabled | 9h session 制限対応 |
| Storage cap | 18 GB | /kaggle/working 安全マージン |

## 期待規模

- 234 species × avg 50-200 recordings/species = 12k-50k 録音
- Quality A/B/C filter で ~10-30k
- 容量 ~10-30 GB

## 完了後

`maekeso/birdclef2026-xc-api-aves-audio` 等として Kaggle Dataset 化


In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
import os, time, json, sys
from pathlib import Path
import pandas as pd
import requests
from tqdm.auto import tqdm

# ★ XC API v3 (key 必須、v2 は deprecated)
# This NB is PRIVATE — key embedded directly. Keep NB private to protect key.
XC_API_KEY = "1eef1b2770d218e80cea18fc28621e8dcad01f6a"
XC_API_URL = "https://xeno-canto.org/api/3/recordings"

# Self-contained: species list 埋め込み (taxonomy.csv 不要)
OUT_DIR = Path("/kaggle/working/xc_api_part3")  # ★ Part 3: non-Aves
OUT_DIR.mkdir(exist_ok=True, parents=True)
AUDIO_DIR = OUT_DIR / "audio"
AUDIO_DIR.mkdir(exist_ok=True, parents=True)
URL_CACHE = OUT_DIR / "_urls.csv"  # query result cache for resume

# Politeness config
QUERY_DELAY_SEC = 1.5        # XC API query 間隔
DL_DELAY_SEC = 1.5           # audio DL 間隔
REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
RETRY_BACKOFF_SEC = [5, 15, 60]
USER_AGENT = (
    "BirdCLEF2026-research/1.0 "
    "(Kaggle research project; contact: kaggle.com/maekeso)"
)

# Filters
QUALITY_FILTER = ["A", "B", "C"]  # high-medium quality
MAX_LENGTH_SEC = 120              # ≤2 min only
EXCLUDE_ND_LICENSE = True         # No-Derivatives 除外 (再配布不可なため)
MAX_PER_SPECIES_QUERY = 500       # XC API 1 page = 500、複数 page 取得可
MAX_PER_SPECIES_DL = None         # None = all, set int to cap

# Safety
STORAGE_CAP_GB = 18.0
MAX_RUNTIME_HOURS = 8.5

print(f"Output: {OUT_DIR}")
print(f"Polite delays: query={QUERY_DELAY_SEC}s, dl={DL_DELAY_SEC}s")
print(f"Filters: quality={QUALITY_FILTER}, length<={MAX_LENGTH_SEC}s, exclude_ND={EXCLUDE_ND_LICENSE}")


In [ ]:
# ============================================================
# Cell 2: BC2026 non-Aves species list (embedded, 72 species)
# ============================================================
__BC2026_NON_AVES_RAW = [
    ('Guyalna cuta', '1161364', 'Insecta'),
    ('Caiman yacare', '116570', 'Reptilia'),
    ('Leptodactylus luctator', '1176823', 'Amphibia'),
    ('Adenomera guarani', '1491113', 'Amphibia'),
    ('Lysapsus limellum', '1595929', 'Amphibia'),
    ('Equus caballus', '209233', 'Mammalia'),
    ('Leptodactylus syphax', '22930', 'Amphibia'),
    ('Leptodactylus mystacinus', '22956', 'Amphibia'),
    ('Leptodactylus podicipinus', '22961', 'Amphibia'),
    ('Leptodactylus elenae', '22967', 'Amphibia'),
    ('Leptodactylus fuscus', '22973', 'Amphibia'),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia'),
    ('Leptodactylus petersii', '22985', 'Amphibia'),
    ('Physalaemus centralis', '23150', 'Amphibia'),
    ('Physalaemus albifrons', '23154', 'Amphibia'),
    ('Physalaemus albonotatus', '23158', 'Amphibia'),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia'),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia'),
    ('Scinax nasicus', '24279', 'Amphibia'),
    ('Scinax fuscovarius', '24285', 'Amphibia'),
    ('Scinax fuscomarginatus', '24287', 'Amphibia'),
    ('Scinax acuminatus', '24321', 'Amphibia'),
    ('Quesada gigas', '244024', 'Insecta'),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia'),
    ('Elachistocleis bicolor', '25092', 'Amphibia'),
    ('Dermatonotus muelleri', '25214', 'Amphibia'),
    ('Physalaemus biligonigerus', '326272', 'Amphibia'),
    ('Panthera onca', '41970', 'Mammalia'),
    ('Alouatta caraya', '43435', 'Mammalia'),
    ('Canis familiaris', '47144', 'Mammalia'),
    ('Insect son01', '47158son01', 'Insecta'),
    ('Insect son02', '47158son02', 'Insecta'),
    ('Insect son03', '47158son03', 'Insecta'),
    ('Insect son04', '47158son04', 'Insecta'),
    ('Insect son05', '47158son05', 'Insecta'),
    ('Insect son06', '47158son06', 'Insecta'),
    ('Insect son07', '47158son07', 'Insecta'),
    ('Insect son08', '47158son08', 'Insecta'),
    ('Insect son09', '47158son09', 'Insecta'),
    ('Insect son10', '47158son10', 'Insecta'),
    ('Insect son11', '47158son11', 'Insecta'),
    ('Insect son12', '47158son12', 'Insecta'),
    ('Insect son13', '47158son13', 'Insecta'),
    ('Insect son14', '47158son14', 'Insecta'),
    ('Insect son15', '47158son15', 'Insecta'),
    ('Insect son16', '47158son16', 'Insecta'),
    ('Insect son17', '47158son17', 'Insecta'),
    ('Insect son18', '47158son18', 'Insecta'),
    ('Insect son19', '47158son19', 'Insecta'),
    ('Insect son20', '47158son20', 'Insecta'),
    ('Insect son21', '47158son21', 'Insecta'),
    ('Insect son22', '47158son22', 'Insecta'),
    ('Insect son23', '47158son23', 'Insecta'),
    ('Insect son24', '47158son24', 'Insecta'),
    ('Insect son25', '47158son25', 'Insecta'),
    ('Physalaemus nattereri', '476521', 'Amphibia'),
    ('Sapajus cay', '516975', 'Mammalia'),
    ('Pithecopus azureus', '517063', 'Amphibia'),
    ('Boana lundii', '555123', 'Amphibia'),
    ('Boana punctata', '555145', 'Amphibia'),
    ('Boana raniceps', '555146', 'Amphibia'),
    ('Ameerega picta', '64898', 'Amphibia'),
    ('Dendropsophus minutus', '65377', 'Amphibia'),
    ('Dendropsophus nanus', '65380', 'Amphibia'),
    ('Pseudis platensis', '66971', 'Amphibia'),
    ('Rhinella diptycha', '67107', 'Amphibia'),
    ('Trachycephalus typhonius', '67252', 'Amphibia'),
    ('Leptodactylus macrosternum', '70711', 'Amphibia'),
    ('Plecturocebus pallescens', '738183', 'Mammalia'),
    ('Bos taurus', '74113', 'Mammalia'),
    ('Mico melanurus', '74580', 'Mammalia'),
    ('Prionacris erosa', '760266', 'Insecta'),
]

# Convert to df with columns matching Stage 1 pipeline
aves_df = pd.DataFrame(__BC2026_NON_AVES_RAW, columns=["scientific_name", "primary_label", "class_name"])
print(f"BC2026 non-Aves species: {len(aves_df)}")
print(f"  classes: {aves_df['class_name'].value_counts().to_dict()}")
print(aves_df.head().to_string(index=False))

def parse_sci_name(name):
    parts = str(name).strip().split()
    if len(parts) >= 2:
        return parts[0], " ".join(parts[1:])
    return parts[0], ""

aves_df[["genus", "species"]] = aves_df["scientific_name"].apply(
    lambda x: pd.Series(parse_sci_name(x))
)
print(f"\nUnique genera: {aves_df['genus'].nunique()}")

# No skip list (Part 3 = fresh non-Aves, no prior runs)


In [ ]:
# ============================================================
# Cell 3: Query XC API for each species → build URL list (with cache)
# ============================================================
# XC API v3: https://xeno-canto.org/api/3/recordings?key=...&query=...
# Returns JSON with `recordings` list and pagination info (v2 was deprecated, 404)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

if URL_CACHE.exists():
    print(f"Loading cached URL list: {URL_CACHE}")
    url_df = pd.read_csv(URL_CACHE)
    print(f"  cached: {len(url_df)} URLs from {url_df['scientific_name'].nunique()} species")
    queried_species = set(url_df["scientific_name"].unique())
else:
    url_df = pd.DataFrame()
    queried_species = set()

records = [url_df] if len(url_df) else []
species_to_query = [s for s in aves_df["scientific_name"].tolist() if s not in queried_species]
print(f"\nSpecies to query: {len(species_to_query)} (of {len(aves_df)})")

for idx, sci_name in enumerate(tqdm(species_to_query, desc="XC query")):
    genus, species = parse_sci_name(sci_name)
    species_records = []
    page = 1
    while True:
        # XC API v3 query (key required)
        q = f"gen:{genus} sp:{species}"
        params = {"query": q, "key": XC_API_KEY, "page": page}
        try:
            r = session.get(XC_API_URL, params=params, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f"\n  [{sci_name}] page {page} query err: {str(e)[:150]}")
            break

        recs = data.get("recordings", [])
        if not recs:
            break

        for rec in recs:
            # v3 schema (likely same as v2 but some fields may differ)
            species_records.append({
                "scientific_name": sci_name,
                "xc_id": rec.get("id"),
                "genus": rec.get("gen"),
                "species": rec.get("sp"),
                "subspecies": rec.get("ssp"),
                "en_name": rec.get("en"),
                "type": rec.get("type"),
                "file_url": rec.get("file") or rec.get("file-name") or rec.get("download"),
                "quality": rec.get("q"),
                "length_str": rec.get("length"),
                "country": rec.get("cnt"),
                "license": rec.get("lic"),
                "rmk": rec.get("rmk"),
            })

        num_pages = int(data.get("numPages", 1))
        if page >= num_pages:
            break
        page += 1
        time.sleep(QUERY_DELAY_SEC)  # polite between pages

    records.append(pd.DataFrame(species_records))
    time.sleep(QUERY_DELAY_SEC)  # polite between species

    # Periodic save (every 20 species)
    if (idx + 1) % 20 == 0:
        url_df_partial = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
        if len(url_df_partial):
            url_df_partial.to_csv(URL_CACHE, index=False)
            print(f"\n  [{idx+1}/{len(species_to_query)}] saved {len(url_df_partial)} URLs to cache")

# Final concat + save
url_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
url_df.to_csv(URL_CACHE, index=False)

print(f"\nFinal URL collection: {len(url_df)} URLs")
print(f"  from {url_df['scientific_name'].nunique()} species")
print(f"  quality dist: {url_df['quality'].value_counts().to_dict()}")


In [ ]:
# ============================================================
# Cell 4: Filter URLs (quality, length, license)
# ============================================================
def length_str_to_sec(s):
    """XC length format 'MM:SS' or 'M:SS' → seconds."""
    try:
        parts = str(s).split(":")
        if len(parts) == 2:
            return int(parts[0]) * 60 + int(parts[1])
        elif len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    except Exception:
        return None
    return None

url_df["length_sec"] = url_df["length_str"].apply(length_str_to_sec)
print(f"Total URLs: {len(url_df)}")

# Filter chain
n = len(url_df)
url_df = url_df.dropna(subset=["file_url", "xc_id"])
print(f"  after dropna(file_url/xc_id): {n} -> {len(url_df)}")

n = len(url_df)
url_df = url_df[url_df["quality"].isin(QUALITY_FILTER)].reset_index(drop=True)
print(f"  after quality filter {QUALITY_FILTER}: {n} -> {len(url_df)}")

n = len(url_df)
url_df = url_df[url_df["length_sec"].notna() & (url_df["length_sec"] <= MAX_LENGTH_SEC)].reset_index(drop=True)
print(f"  after length<={MAX_LENGTH_SEC}s: {n} -> {len(url_df)}")

if EXCLUDE_ND_LICENSE:
    n = len(url_df)
    # XC license typically contains "BY-NC-ND" or "BY-NC-SA" or "BY"
    is_nd = url_df["license"].astype(str).str.contains("nd", case=False, na=False)
    url_df = url_df[~is_nd].reset_index(drop=True)
    print(f"  after exclude ND license: {n} -> {len(url_df)}")

if MAX_PER_SPECIES_DL:
    n = len(url_df)
    url_df = url_df.groupby("scientific_name", group_keys=False).head(MAX_PER_SPECIES_DL).reset_index(drop=True)
    print(f"  after per-species cap {MAX_PER_SPECIES_DL}: {n} -> {len(url_df)}")

print(f"\n=== After all filters: {len(url_df)} URLs ===")
print(f"  species covered: {url_df['scientific_name'].nunique()} / {len(aves_df)}")
print(f"  quality dist: {url_df['quality'].value_counts().to_dict()}")
print(f"  est total length: {url_df['length_sec'].sum()/3600:.1f}h")

url_df.to_csv(OUT_DIR / "_url_filtered.csv", index=False)


In [ ]:
# ============================================================
# Cell 5: Polite audio download (resume + safety limits)
# ============================================================
def safe_species_dir(name):
    return str(name).replace(" ", "_").replace("/", "_").replace("'", "")

url_df["species_dir"] = url_df["scientific_name"].apply(safe_species_dir)
url_df["save_path"] = url_df.apply(
    lambda r: AUDIO_DIR / r["species_dir"] / f"XC{r['xc_id']}.mp3",
    axis=1,
)

# Resume detection
url_df["already_dl"] = url_df["save_path"].apply(
    lambda p: p.exists() and p.stat().st_size > 1000
)
n_existing = int(url_df["already_dl"].sum())
n_to_dl = len(url_df) - n_existing
print(f"Already downloaded: {n_existing}/{len(url_df)}")
print(f"To download: {n_to_dl}")

stats = {
    "ok": 0, "skip_existing": n_existing,
    "fail_total": 0, "fail_4xx": 0, "fail_5xx": 0,
    "fail_timeout": 0, "fail_other": 0,
    "bytes_dl_session": 0,
    "fail_urls": [],
}
start_t = time.time()
to_dl = url_df[~url_df["already_dl"]].reset_index(drop=True)
print(f"Estimated DL time: {len(to_dl) * DL_DELAY_SEC / 3600:.1f}h (at {DL_DELAY_SEC}s/req)")

SAVE_EVERY = 200

def save_progress_state():
    state_df = url_df[["xc_id", "scientific_name", "file_url", "save_path", "already_dl"]].copy()
    state_df["save_path"] = state_df["save_path"].astype(str)
    state_df["downloaded_now"] = state_df["save_path"].apply(
        lambda p: Path(p).exists() and Path(p).stat().st_size > 1000
    )
    state_df.to_csv(OUT_DIR / "_dl_state.csv", index=False)

try:
    for idx, row in tqdm(to_dl.iterrows(), total=len(to_dl), desc="DL"):
        # Safety: runtime
        elapsed_h = (time.time() - start_t) / 3600
        if elapsed_h > MAX_RUNTIME_HOURS:
            print(f"\n⚠ Hit runtime cap {MAX_RUNTIME_HOURS}h, stopping. Resume next session.")
            break

        # Safety: storage (poll every 50 files)
        if (idx + 1) % 50 == 0:
            try:
                total_gb = sum(f.stat().st_size for f in AUDIO_DIR.rglob("*.mp3")) / 1e9
                if total_gb > STORAGE_CAP_GB:
                    print(f"\n⚠ Storage cap reached ({total_gb:.2f} GB > {STORAGE_CAP_GB}). Stopping.")
                    break
            except Exception:
                pass

        save_path = Path(row["save_path"])
        save_path.parent.mkdir(parents=True, exist_ok=True)
        url = row["file_url"]

        success = False
        last_err = None
        for attempt in range(MAX_RETRIES):
            try:
                r = session.get(url, timeout=REQUEST_TIMEOUT, stream=True)
                if r.status_code == 200:
                    with open(save_path, "wb") as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
                    sz = save_path.stat().st_size
                    if sz < 1000:
                        save_path.unlink()
                        last_err = f"too small ({sz}B)"
                        raise ValueError(last_err)
                    stats["ok"] += 1
                    stats["bytes_dl_session"] += sz
                    success = True
                    break
                elif 400 <= r.status_code < 500:
                    last_err = f"HTTP {r.status_code}"
                    stats["fail_4xx"] += 1
                    break  # 4xx は retry 無効
                elif 500 <= r.status_code < 600:
                    last_err = f"HTTP {r.status_code}"
                    stats["fail_5xx"] += 1
            except requests.Timeout:
                last_err = "timeout"
                stats["fail_timeout"] += 1
            except Exception as e:
                last_err = str(e)[:100]
                stats["fail_other"] += 1

            if not success and attempt < MAX_RETRIES - 1:
                wait = RETRY_BACKOFF_SEC[min(attempt, len(RETRY_BACKOFF_SEC) - 1)]
                time.sleep(wait)

        if not success:
            stats["fail_total"] += 1
            stats["fail_urls"].append((url, last_err))
            if save_path.exists():
                try: save_path.unlink()
                except: pass

        time.sleep(DL_DELAY_SEC)

        if (idx + 1) % SAVE_EVERY == 0:
            elapsed = time.time() - start_t
            rate = (idx + 1) / elapsed if elapsed > 0 else 0
            remain_s = (len(to_dl) - idx - 1) / rate if rate > 0 else 0
            total_gb = stats["bytes_dl_session"] / 1e9
            print(f"  [{idx+1}/{len(to_dl)}] ok={stats['ok']} fail={stats['fail_total']} "
                  f"sess_GB={total_gb:.2f} rate={rate*60:.1f}/min ETA={remain_s/3600:.1f}h")
            save_progress_state()

except KeyboardInterrupt:
    print("\n⚠ Interrupted")
except Exception as e:
    print(f"\n⚠ Error: {e}")
    import traceback; traceback.print_exc()

save_progress_state()
print(f"\n{'='*60}\nDL session summary\n{'='*60}")
for k, v in stats.items():
    if k != "fail_urls":
        print(f"  {k}: {v}")
print(f"  Session time: {(time.time()-start_t)/60:.1f} min")


In [ ]:
# ============================================================
# Cell 6: Verify + final metadata
# ============================================================
url_df["downloaded"] = url_df["save_path"].apply(
    lambda p: Path(p).exists() and Path(p).stat().st_size > 1000
)
url_df["file_size_mb"] = url_df["save_path"].apply(
    lambda p: Path(p).stat().st_size / 1e6 if Path(p).exists() else 0.0
)

n_dl = int(url_df["downloaded"].sum())
total_gb = url_df["file_size_mb"].sum() / 1024
print(f"=== Final state ===")
print(f"  Total URLs: {len(url_df)}")
print(f"  Downloaded: {n_dl} ({100*n_dl/len(url_df):.1f}%)")
print(f"  Total size: {total_gb:.2f} GB")

sp_summary = url_df.groupby("scientific_name").agg(
    total_urls=("xc_id", "count"),
    downloaded=("downloaded", "sum"),
    total_mb=("file_size_mb", "sum"),
).reset_index()
sp_summary["completion_pct"] = 100 * sp_summary["downloaded"] / sp_summary["total_urls"]
sp_summary = sp_summary.sort_values("completion_pct", ascending=False)

print(f"\n=== Per-species coverage (top 5) ===")
print(sp_summary.head().to_string(index=False))
print(f"\n=== Per-species coverage (bottom 5) ===")
print(sp_summary.tail().to_string(index=False))

fully = (sp_summary["completion_pct"] == 100).sum()
partial = ((sp_summary["completion_pct"] < 100) & (sp_summary["completion_pct"] > 0)).sum()
none = (sp_summary["completion_pct"] == 0).sum()
print(f"\n  Fully downloaded: {fully}/{len(sp_summary)} species")
print(f"  Partial: {partial}")
print(f"  Not started: {none}")

# Save final metadata
final_meta = url_df[url_df["downloaded"]][[
    "xc_id", "scientific_name", "genus", "species", "en_name", "type",
    "quality", "length_sec", "country", "license", "save_path", "file_size_mb"
]].copy()
final_meta["filename"] = final_meta["save_path"].apply(
    lambda p: str(Path(p).relative_to(AUDIO_DIR))
)
final_meta = final_meta.drop(columns=["save_path"])
final_meta.to_csv(OUT_DIR / "_metadata.csv", index=False)
sp_summary.to_csv(OUT_DIR / "_species_summary.csv", index=False)

print(f"\nSaved: {OUT_DIR / '_metadata.csv'}")
print(f"Saved: {OUT_DIR / '_species_summary.csv'}")


In [ ]:
# ============================================================
# Cell 7: (Optional) Save as Kaggle Dataset
# ============================================================
import json, shutil
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

SLUG = "birdclef2026-xc-api-aves-audio-part3"
TITLE = "BirdCLEF2026 XC API Aves Audio"
USER = "maekeso"

DRY_RUN = True  # ★ confirm before upload

if not DRY_RUN:
    meta = {
        "title": TITLE,
        "id": f"{USER}/{SLUG}",
        "licenses": [{"name": "CC-BY-NC-SA-4.0"}],
    }
    (OUT_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

    try:
        api.dataset_create_version(folder=str(OUT_DIR), version_notes="initial XC API DL",
                                    dir_mode="zip", quiet=False)
        print("OK new version uploaded")
    except Exception:
        try:
            api.dataset_create_new(folder=str(OUT_DIR), public=False,
                                    dir_mode="zip", quiet=False)
            print("OK new dataset created")
        except Exception as e:
            print(f"upload err: {str(e)[:300]}")
    print(f"URL: https://www.kaggle.com/datasets/{USER}/{SLUG}")
else:
    print(f"DRY_RUN=True, skip upload")
    print(f"To upload: set DRY_RUN=False and re-run this cell")
    print(f"Will upload {n_dl} files ({total_gb:.2f} GB) to {USER}/{SLUG}")
